In [6]:
# create daily classification fit for predictor features - insitu observation

# class 0 - no rain - < 0.1 mm
# class 1 - rain - >= 0.1 mm

# WORKFLOW
# 1. Load dfs and format for fitting
# 2. Perform cluster*season wise ML fit
# 3. Check if cluster stratification improves the model
# 4. Check if season stratification improves the model 
# 5. Saved trained regressor model using joblib


## REMOVE FEATURES

In [7]:
# Libraries
import os
import numpy as np  
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from pathlib import Path

import operator
import scores.categorical

# Paths
train_df_path = r"..\Data\Processed\Bias_correction_daily\Prepared_dataframe\final_train_df.parquet"
test_df_path = r"..\Data\Processed\Bias_correction_daily\Prepared_dataframe\final_test_df.parquet"
classifier_model_path = r"..\Data\Processed\Bias_correction_daily\Classifier_model\saved_classifiers"
classifier_metrics_path = r"..\Data\Processed\Bias_correction_daily\Classifier_model\validation_results"
rscv_dir = r"..\Data\Processed\Bias_correction_daily\Classifier_model\rscv_results"

os.makedirs(classifier_model_path, exist_ok = True)
os.makedirs(classifier_metrics_path, exist_ok = True)
os.makedirs(rscv_dir, exist_ok = True)

# Configurations
random_state_num = 64
n_cores = 4 
target_variable = 'insitu_prec_class'
predictor_features = ['y', 'x', 
        'era5l', 'chirps', 'har', 
        'persiann_prec', 'mswep_prec', 
        'gldas_air_temp', 'gldas_lwdown', 'gldas_specific_humidity',
        'gldas_surface_pressure', 'gldas_swdown', 'gldas_wind_speed',
        'insitu_kriged', 'cluster', 'season', 'season_num',
        'elev_mean', 'elev_std', 'elev_max', 'elev_min', 'slope_mean', 
        'aspect_mean', 'elev_smoothed', 'fetch_1000m_dist', 'fetch_1000m_dir',
        'fetch_500m_dist', 'fetch_500m_dir', 'local_relief', 'doy_sin', 'doy_cos']

In [8]:
SEASONS = {
    'winter':       [12, 1, 2],
    'pre_monsoon':  [3, 4, 5],
    'monsoon':      [6, 7, 8, 9],
    'post_monsoon': [10, 11],
}

def get_season(month):
    for s, months in SEASONS.items():
        if month in months:
            return s
    return None

SEASONS_NUM = {
    1: 'winter',
    2: 'pre_monsoon',
    3: 'monsoon',
    4: 'post_monsoon'
}

def get_season_num(season_str):
    for s_n, s_name in SEASONS_NUM.items():
        if s_name == season_str:
            return s_n
    return None

# Helper functions
def get_param_grids():
    param_dist_rf = {
        'n_estimators':      [100, 500, 1000, 2000, 3000],
        'max_features':      ['sqrt', 'log2'],
        'max_depth':         [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
    }
    param_dist_xgb = {
        'n_estimators':     [100, 500, 1000, 2000, 3000],
        'max_depth':        [3, 5, 7, 10],
        'learning_rate':    [0.01, 0.05, 0.1, 0.2],
        'subsample':        [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma':            [0, 0.1, 0.5],
    }
    
    return param_dist_rf, param_dist_xgb

def compute_metrics_classification(y_true, y_pred, label):
    """ Compute categorical metrices"""
    #define event -  wet day is when precipitation is greater than 0.1
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred) 
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    print(f"[{label}] | accuracy = {accuracy:.02f}| f1-score = {f1:.02f}| n = {len(y_true)}")
    return {'accuracy': round(accuracy, 3), 'f1-score': round(f1, 3), 'n': len(y_true)}

In [9]:
# 1. Load and format dfs
train_df = pd.read_parquet(train_df_path)
test_df = pd.read_parquet(test_df_path)

seasons_train_da = train_df['month'].apply(get_season)
train_df['season'] = seasons_train_da
seasons_test_da = test_df['month'].apply(get_season)
test_df['season'] = seasons_test_da

train_df['season_num'] = train_df['season'].apply(get_season_num)
test_df['season_num'] = test_df['season'].apply(get_season_num)

train_df['insitu_prec_class'] = np.where(train_df['insitu_prec'] >= 0.1, 1, 0)
test_df['insitu_prec_class'] = np.where(test_df['insitu_prec'] >= 0.1, 1, 0)

cluster_list = sorted(test_df['cluster'].unique().tolist())
season_list = list(SEASONS)

In [10]:
# 2. First we perform a full cluster*season stratified fit

for cluster in cluster_list:
    cluster_train = train_df[train_df['cluster'] == cluster].copy()
    cluster_test = test_df[test_df['cluster'] == cluster].copy()

    for season in season_list:
        print(f"Processing Cluster {cluster} & Season {season} -----------------------------")

        season_cluster_train = cluster_train[cluster_train['season'] == season].copy()
        season_cluster_test = cluster_test[cluster_test['season'] == season].copy()
        
        X_train = season_cluster_train[predictor_features].drop(columns= ['cluster', 'season', 'season_num'], errors='ignore').values
        y_train = season_cluster_train[target_variable].values

        param_dist_rf, param_dist_xgb = get_param_grids()


        for model_type in ['rf', 'xgb']:
            if model_type == 'rf':
                base_model = Pipeline([
                    ('imputer', SimpleImputer(strategy = 'median')),
                    ('classifier', RandomForestClassifier(random_state = random_state_num,
                        n_jobs = n_cores))
                ])
                param_dist = {f'classifier__{k}': v for k, v in param_dist_rf.items()}
            else:
                base_model = Pipeline([
                    ('imputer', SimpleImputer(strategy = 'median')),
                    ('classifier',     XGBClassifier(random_state=random_state_num,
                                    n_jobs=n_cores,
                                    verbosity = 0)),    
                ])
                param_dist = {f'classifier__{k}': v for k, v in param_dist_xgb.items()}

            search_cv = RandomizedSearchCV(
                estimator = base_model,
                param_distributions = param_dist,
                n_iter = 50,
                cv = 5,
                verbose = 1,
                n_jobs = n_cores,
                scoring = 'f1'
            )
            search_cv.fit(X_train, y_train)
            best_model = search_cv.best_estimator_
            rscv_results_df = pd.DataFrame(search_cv.cv_results_)
            rscv_path = os.path.join(rscv_dir, f"rscv_result_{cluster}_{season}_{model_type}.csv")
            rscv_results_df.to_csv(rscv_path)

            print(f"\n  [Cluster {cluster} | {model_type.upper()}]")
            print(f"    Best params : {search_cv.best_params_}")
            print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
            print(f"RSCV results saved to : {rscv_path}")
            
            model_path = os.path.join(classifier_model_path, f"{cluster}_{season}_{model_type}.pkl")
            joblib.dump(best_model, model_path)
            print(f"Model  {model_type} saved to {model_path}")

Processing Cluster 0.0 & Season winter -----------------------------
Fitting 5 folds for each of 50 candidates, totalling 250 fits

  [Cluster 0.0 | RF]
    Best params : {'classifier__n_estimators': 3000, 'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 4, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 20}
    CV RMSE     : 0.6654
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Classifier_model\rscv_results\rscv_result_0.0_winter_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Classifier_model\saved_classifiers\0.0_winter_rf.pkl
Fitting 5 folds for each of 50 candidates, totalling 250 fits

  [Cluster 0.0 | XGB]
    Best params : {'classifier__subsample': 0.8, 'classifier__n_estimators': 500, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.01, 'classifier__gamma': 0, 'classifier__colsample_bytree': 0.6}
    CV RMSE     : 0.6658
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Classifier_model

In [11]:
# 3. Cluster wise only fit

for cluster in cluster_list:
    print(f"Processing Cluster {cluster}-----------------------------")
    cluster_train = train_df[train_df['cluster'] == cluster].copy()
    cluster_test = test_df[test_df['cluster'] == cluster].copy()
        
    X_train = cluster_train[predictor_features].drop(columns= ['cluster', 'season'], errors='ignore').values
    y_train = cluster_train[target_variable].values

    param_dist_rf, param_dist_xgb = get_param_grids()

    for model_type in ['rf', 'xgb']:
        if model_type == 'rf':
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('classifier', RandomForestClassifier(random_state = random_state_num,
                    n_jobs = n_cores))
            ])
            param_dist = {f'classifier__{k}': v for k, v in param_dist_rf.items()}
        else:
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('classifier',     XGBClassifier(random_state=random_state_num,
                                n_jobs=n_cores,
                                verbosity = 0)),    
            ])
            param_dist = {f'classifier__{k}': v for k, v in param_dist_xgb.items()}

        search_cv = RandomizedSearchCV(
            estimator = base_model,
            param_distributions = param_dist,
            n_iter = 50,
            cv = 5,
            verbose = 1,
            n_jobs = n_cores,
            scoring = 'f1'
        )
        search_cv.fit(X_train, y_train)
        best_model = search_cv.best_estimator_
        rscv_results_df = pd.DataFrame(search_cv.cv_results_)
        rscv_path = os.path.join(rscv_dir, f"rscv_result_{cluster}_{model_type}.csv")
        rscv_results_df.to_csv(rscv_path)

        print(f"\n  [Cluster {cluster} | {model_type.upper()}]")
        print(f"    Best params : {search_cv.best_params_}")
        print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
        print(f"RSCV results saved to : {rscv_path}")
            
        model_path = os.path.join(classifier_model_path, f"{cluster}_{model_type}.pkl")
        joblib.dump(best_model, model_path)
        print(f"Model  {model_type} saved to {model_path}")

Processing Cluster 0.0-----------------------------
Fitting 5 folds for each of 50 candidates, totalling 250 fits

  [Cluster 0.0 | RF]
    Best params : {'classifier__n_estimators': 100, 'classifier__min_samples_split': 2, 'classifier__min_samples_leaf': 2, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 10}
    CV RMSE     : 0.6874
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Classifier_model\rscv_results\rscv_result_0.0_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Classifier_model\saved_classifiers\0.0_rf.pkl
Fitting 5 folds for each of 50 candidates, totalling 250 fits

  [Cluster 0.0 | XGB]
    Best params : {'classifier__subsample': 1.0, 'classifier__n_estimators': 1000, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.01, 'classifier__gamma': 0.5, 'classifier__colsample_bytree': 0.6}
    CV RMSE     : 0.6914
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Classifier_model\rscv_results\rscv_result_0.0_

In [ ]:
# 3. Season wise only fit
for season in season_list:
    print(f"\n Processing Season {season}-----------------------------")
    season_train = train_df[train_df['season'] == season].copy()
    season_test = test_df[test_df['season'] == season].copy()
            
    X_train = season_train[predictor_features].drop(columns= ['season', 'season_num'], errors='ignore').values
    y_train = season_train[target_variable].values

    param_dist_rf, param_dist_xgb = get_param_grids()

    for model_type in ['rf', 'xgb']:
        if model_type == 'rf':
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('classifier', RandomForestClassifier(random_state = random_state_num,
                    n_jobs = n_cores))
            ])
            param_dist = {f'classifier__{k}': v for k, v in param_dist_rf.items()}
        else:
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('classifier',     XGBClassifier(random_state=random_state_num,
                                n_jobs=n_cores,
                                verbosity = 0)),    
            ])
            param_dist = {f'classifier__{k}': v for k, v in param_dist_xgb.items()}

        search_cv = RandomizedSearchCV(
            estimator = base_model,
            param_distributions = param_dist,
            n_iter = 50,
            cv = 5,
            verbose = 1,
            n_jobs = n_cores,
            scoring = 'f1'
        )
        search_cv.fit(X_train, y_train)
        best_model = search_cv.best_estimator_
        rscv_results_df = pd.DataFrame(search_cv.cv_results_)
        rscv_path  = os.path.join(rscv_dir, f"rscv_result_{season}_{model_type}.csv")
        rscv_results_df.to_csv(rscv_path)

        print(f"\n  [Cluster {season} | {model_type.upper()}]")
        print(f"    Best params : {search_cv.best_params_}")
        print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
        print(f"RSCV results saved to : {rscv_path}")
                
        model_path = os.path.join(classifier_model_path, f"{season}_{model_type}.pkl")
        joblib.dump(best_model, model_path)
        print(f"Model  {model_type} saved to {model_path}")




 Processing Season winter-----------------------------
Fitting 5 folds for each of 50 candidates, totalling 250 fits

  [Cluster winter | RF]
    Best params : {'classifier__n_estimators': 3000, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 4, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 20}
    CV RMSE     : 0.6041
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Classifier_model\rscv_results\rscv_result_winter_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Classifier_model\saved_classifiers\winter_rf.pkl
Fitting 5 folds for each of 50 candidates, totalling 250 fits

  [Cluster winter | XGB]
    Best params : {'classifier__subsample': 0.6, 'classifier__n_estimators': 100, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.1, 'classifier__gamma': 0.5, 'classifier__colsample_bytree': 1.0}
    CV RMSE     : 0.6144
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Classifier_model\rscv_results\r

In [ ]:
# 4. Aggregated fit

train = train_df.copy()
test = test_df.copy()
        
X_train = train[predictor_features].drop(columns= ['season'], errors='ignore').values
y_train = train[target_variable].values

param_dist_rf, param_dist_xgb = get_param_grids()

for model_type in ['rf', 'xgb']:
    if model_type == 'rf':
        base_model = Pipeline([
            ('imputer', SimpleImputer(strategy = 'median')),
            ('classifier', RandomForestClassifier(random_state = random_state_num,
                n_jobs = n_cores))
        ])
        param_dist = {f'classifier__{k}': v for k, v in param_dist_rf.items()}
    else:
        base_model = Pipeline([
            ('imputer', SimpleImputer(strategy = 'median')),
            ('classifier',     XGBClassifier(random_state=random_state_num,
                            n_jobs=n_cores,
                            verbosity = 0)),    
        ])
        param_dist = {f'classifier__{k}': v for k, v in param_dist_xgb.items()}

    search_cv = RandomizedSearchCV(
        estimator = base_model,
        param_distributions = param_dist,
        n_iter = 50,
        cv = 5,
        verbose = 1,
        n_jobs = n_cores,
        scoring = 'f1'
    )
    search_cv.fit(X_train, y_train)
    best_model = search_cv.best_estimator_
    rscv_results_df = pd.DataFrame(search_cv.cv_results_)
    rscv_path = os.path.join(rscv_dir, f"rscv_result_aggregated_{model_type}.csv")
    rscv_results_df.to_csv(rscv_path)

    model_path = os.path.join(classifier_model_path, f"Aggregated_{model_type}.pkl")
    joblib.dump(best_model, model_path)

    print(f"{model_type.upper()}]")
    print(f"    Best params : {search_cv.best_params_}")
    print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
    print(f"RSCV results saved to : {rscv_path}")
    print(f"Model  {model_type} saved to {model_path}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
RF]
    Best params : {'classifier__n_estimators': 1000, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 4, 'classifier__max_features': 'sqrt', 'classifier__max_depth': 20}
    CV RMSE     : -0.3974
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Classifier_model\rscv_results\rscv_result_aggregated_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Classifier_model\saved_classifiers\Aggregated_rf.pkl
Fitting 5 folds for each of 50 candidates, totalling 250 fits
XGB]
    Best params : {'classifier__subsample': 0.6, 'classifier__n_estimators': 2000, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.01, 'classifier__gamma': 0.1, 'classifier__colsample_bytree': 0.8}
    CV RMSE     : -0.3951
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Classifier_model\rscv_results\rscv_result_aggregated_xgb.csv
Model  xgb saved to ..\Data\Processed\Bias_correction_da

In [ ]:
# %%
# 5. Validation: per cluster*season subset, compare all four model strategies

# Feature subsets matching how each strategy was trained
feature_sets = {
    'cluster_season': [f for f in predictor_features if f not in ('cluster', 'season', 'season_num')],
    'cluster_only':   [f for f in predictor_features if f not in ('cluster', 'season')],
    'season_only':    [f for f in predictor_features if f not in ('season', 'season_num')],
    'aggregated':     [f for f in predictor_features if f not in ('season',)],
}

# Model path templates per strategy
def get_model_path(strategy, cluster, season, model_type):
    templates = {
        'cluster_season': f"{cluster}_{season}_{model_type}.pkl",
        'cluster_only':   f"{cluster}_{model_type}.pkl",
        'season_only':    f"{season}_{model_type}.pkl",   # NOTE: season-only loop has a
        'aggregated':     f"Aggregated_{model_type}.pkl", #  bug saving as cluster_{mt}.pkl;
    }                                                      #  fix that loop's save path too
    return os.path.join(classifier_model_path, templates[strategy])

all_metrics = []

for cluster in cluster_list:
    cluster_test = test_df[test_df['cluster'] == cluster].copy()

    for season in season_list:
        season_cluster_test = cluster_test[cluster_test['season'] == season].copy()

        if season_cluster_test.empty:
            print(f"[SKIP] No test data — Cluster {cluster} | Season {season}")
            continue

        y_test = season_cluster_test[target_variable].values

        for model_type in ['rf', 'xgb']:
            for strategy, features in feature_sets.items():
                model_path = get_model_path(strategy, cluster, season, model_type)

                if not os.path.exists(model_path):
                    print(f"  [MISSING] {model_path}")
                    continue

                model    = joblib.load(model_path)
                X_test   = season_cluster_test[features].values
                y_pred   = model.predict(X_test)

                label    = f"C{cluster} | {season:<13} | {model_type.upper()} | {strategy}"
                metrics  = compute_metrics_classification(y_test, y_pred, label)
                metrics.update({
                    'cluster':    cluster,
                    'season':     season,
                    'model_type': model_type,
                    'strategy':   strategy,
                })
                all_metrics.append(metrics)

# %%
# Collate and save
metrics_df = pd.DataFrame(all_metrics)

# Pivot for easy visual comparison: rows = cluster*season, cols = strategy*model_type
pivot = metrics_df.pivot_table(
    index   = ['cluster', 'season'],
    columns = ['model_type', 'strategy'],
    values  = ['accuracy', 'f1-score'],
)
pivot.columns = ['_'.join(c) for c in pivot.columns]
pivot = pivot.reset_index()

metrics_out_path = os.path.join(classifier_metrics_path, "validation_comparison.csv")
pivot_out_path   = os.path.join(classifier_metrics_path, "validation_pivot.csv")

metrics_df.to_csv(metrics_out_path, index=False)
pivot.to_csv(pivot_out_path,        index=False)

print(f"\nAll metrics  → {metrics_out_path}")
print(f"Pivot table  → {pivot_out_path}\n")
print(pivot.to_string(index=False))

[C0.0 | winter        | RF | cluster_season] | accuracy = 0.94| f1-score = 0.62| n = 2510
[C0.0 | winter        | RF | cluster_only] | accuracy = 0.93| f1-score = 0.60| n = 2510
[C0.0 | winter        | RF | season_only] | accuracy = 0.94| f1-score = 0.63| n = 2510
[C0.0 | winter        | RF | aggregated] | accuracy = 0.93| f1-score = 0.61| n = 2510
[C0.0 | winter        | XGB | cluster_season] | accuracy = 0.93| f1-score = 0.62| n = 2510
[C0.0 | winter        | XGB | cluster_only] | accuracy = 0.93| f1-score = 0.62| n = 2510
[C0.0 | winter        | XGB | season_only] | accuracy = 0.93| f1-score = 0.61| n = 2510
[C0.0 | winter        | XGB | aggregated] | accuracy = 0.93| f1-score = 0.64| n = 2510
[C0.0 | pre_monsoon   | RF | cluster_season] | accuracy = 0.83| f1-score = 0.30| n = 2760
[C0.0 | pre_monsoon   | RF | cluster_only] | accuracy = 0.83| f1-score = 0.29| n = 2760
[C0.0 | pre_monsoon   | RF | season_only] | accuracy = 0.84| f1-score = 0.31| n = 2760
[C0.0 | pre_monsoon   | RF | 

c:\Users\A S U S\.conda\envs\geo_stack\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Create a combined identifier for model and strategy
metrics_df['model_strategy'] = metrics_df['model_type'] + ' | ' + metrics_df['strategy']

# 1. Rank models within each (cluster, season) sub-problem
# We rank based on f1-score, accuracy, and a composite score (average of f1 and accuracy)
# ascending=False means the highest score gets rank 1.0
metrics_df['f1_rank'] = metrics_df.groupby(['cluster', 'season'])['f1-score'].rank(method='min', ascending=False)



# 2. Aggregate across all clusters and seasons to find the overall best combination
overall_ranking = metrics_df.groupby('model_strategy').agg(
    mean_f1=('f1-score', 'mean'),
    mean_f1_rank=('f1_rank', 'mean')
).reset_index()

# 3. Sort by mean composite rank (lowest rank number is best)
overall_ranking = overall_ranking.sort_values(by='mean_f1_rank')

# Display the final ranking
print("Overall Model x Strategy Ranking:")
display(overall_ranking)

# Optional: Save the overall ranking alongside the other validation metrics
ranking_out_path = os.path.join(classifier_metrics_path, "overall_model_ranking.csv")
overall_ranking.to_csv(ranking_out_path, index=False)
print(f"\nSaved overall ranking to -> {ranking_out_path}")

Overall Model x Strategy Ranking:


,model_strategy,mean_f1,mean_f1_rank
3,rf | season_only,0.501687,3.2500
0,rf | aggregated,0.511000,3.3750
4,xgb | aggregated,0.509813,3.4375
7,xgb | season_only,0.516875,3.5000
2,rf | cluster_season,0.457562,5.3125
1,rf | cluster_only,0.439437,5.5000
6,xgb | cluster_season,0.423313,5.5625
5,xgb | cluster_only,0.450125,5.6875



Saved overall ranking to -> ..\Data\Processed\Bias_correction_daily\Classifier_model\validation_results\overall_model_ranking.csv
